# Сборка данных


1. Статистика по архивам.
2. Выгрузка 2016 года в CSV по каждому архиву.
3. Склейка в `merged.csv`.

In [ ]:
from collections import Counter
from pathlib import Path

import pandas as pd

DATA = Path.cwd().parent / "data"

## 1. Сырые архивы


In [ ]:
def read_head(path, n=10):
    """Первые n записей архива, metadata развёрнута в обычные колонки."""
    head = pd.read_json(path, lines=True, compression="gzip", nrows=n)
    df = pd.json_normalize(head["metadata"])
    # to_numpy, т.к. индексы разные
    df["text"] = head["text"].to_numpy()
    return df

In [ ]:
def archive_stats(path, chunk_size=100_000):
    """Печатает общую сводку по архиву и возвращает таблицу по годам."""
    lo, hi = None, None  # мин/макс дата
    min_len, max_len = None, None
    total = no_date = total_chars = empty_text = 0
    per_year = Counter()
    per_year_chars = Counter()

    with pd.read_json(path, lines=True, compression="gzip", chunksize=chunk_size) as reader:
        for chunk in reader:
            total += len(chunk)

            raw_dates = pd.to_datetime(
                pd.json_normalize(chunk["metadata"])["date"], errors="coerce"
            )
            lengths = chunk["text"].fillna("").str.len().to_numpy()

            total_chars += int(lengths.sum())
            empty_text += int((lengths == 0).sum())
            min_len = lengths.min() if min_len is None else min(min_len, lengths.min())
            max_len = lengths.max() if max_len is None else max(max_len, lengths.max())

            dates = raw_dates.dropna()
            no_date += len(raw_dates) - len(dates)
            if dates.empty:
                continue

            lo = dates.min() if lo is None else min(lo, dates.min())
            hi = dates.max() if hi is None else max(hi, dates.max())

            pair = pd.DataFrame({"year": raw_dates.dt.year.to_numpy(), "chars": lengths})
            pair = pair.dropna(subset=["year"])
            pair["year"] = pair["year"].astype(int)

            per_year.update(pair["year"].value_counts().to_dict())
            per_year_chars.update(pair.groupby("year")["chars"].sum().to_dict())

    print(f"всего статей:   {total:,}")
    print(f"без даты:   {no_date:,}")
    print(f"диапазон:    {lo.date()} .. {hi.date()}")
    print(f"всего символов: {total_chars:,}")
    print(f"средняя длина:  {total_chars / total:,.0f}")
    print(f"мин / макс:   {min_len:,} / {max_len:,}")
    print(f"пустых текстов: {empty_text:,}")

    by_year = pd.Series(per_year).sort_index()
    by_year.index.name = "year"
    chars_by_year = pd.Series(per_year_chars).sort_index()

    return pd.DataFrame({
        "articles": by_year,
        "share_%": (by_year / by_year.sum() * 100).round(1),
        "chars": chars_by_year,
        "avg_len": (chars_by_year / by_year).round(0),
    })

### Interfax

In [ ]:
read_head(DATA / "INTERFAX_RU.gz")

In [ ]:
archive_stats(DATA / "INTERFAX_RU.gz")

### Lenta.ru

In [ ]:
read_head(DATA / "LENTA_RU.gz")

In [ ]:
archive_stats(DATA / "LENTA_RU.gz")

### Fontanka.ru

In [ ]:
fontanka_head = read_head(DATA / "FONTANKA_RU.gz")
fontanka_head

Как выглядит сам текст статьи:

In [ ]:
fontanka_head.loc[0, "text"]

In [ ]:
archive_stats(DATA / "FONTANKA_RU.gz")

## 2. Выгрузка 2016 года

Из каждого архива статьи с датой на `2016` пишутся в CSV рядом с архивом (`LENTA_RU.gz` -> `lenta_ru_2016.csv`).

In [ ]:
def extract_year(src, year=2016, chunk_size=100_000):
    """Отбирает статьи за указанный год и складывает их в CSV рядом с архивом."""
    src = Path(src)
    dst = src.with_name(f"{src.name.split('.')[0].lower()}_{year}.csv")
    dst.unlink(missing_ok=True)

    prefix = str(year)
    kept = 0
    first = True

    with pd.read_json(src, lines=True, compression="gzip", chunksize=chunk_size) as reader:
        for chunk in reader:
            meta = pd.json_normalize(chunk["metadata"])
            meta["text"] = chunk["text"].to_numpy()

            # дата строкой 2016-02-11, фильтр по префиксу
            part = meta[meta["date"].astype("string").str.startswith(prefix, na=False)]
            if part.empty:
                continue

            # теги через |
            if "tags" in part.columns:
                part = part.assign(
                    tags=part["tags"].apply(lambda t: "|".join(t) if isinstance(t, list) else t)
                )

            part.to_csv(
                dst,
                mode="w" if first else "a",
                header=first,
                index=False,
                encoding="utf-8-sig",  # BOM для Excel
            )
            first = False
            kept += len(part)

    if first:
        raise ValueError(f"за {year} год в {src.name} записей нет")

    print(f"{kept:,} статей за {year} год -> {dst} ({dst.stat().st_size / 2**20:.0f} МБ)")
    return dst

In [ ]:
for name in ["INTERFAX_RU.gz", "LENTA_RU.gz", "FONTANKA_RU.gz"]:
    extract_year(DATA / name)

## 3. Общий корпус

Колонки дата, время, заголовок, текст + `source`.

In [ ]:
def load_2016(name):
    df = pd.read_csv(DATA / f"{name}.csv", dtype={"date": "string", "time": "string"})
    return df[["date", "time", "title", "text"]].assign(source=name)

In [ ]:
fontanka_ru_2016 = load_2016("fontanka_ru_2016")
fontanka_ru_2016.head()

In [ ]:
interfax_ru_2016 = load_2016("interfax_ru_2016")
interfax_ru_2016.head()

In [ ]:
lenta_ru_2016 = load_2016("lenta_ru_2016")
lenta_ru_2016.head()

In [ ]:
merged = pd.concat([fontanka_ru_2016, lenta_ru_2016, interfax_ru_2016], ignore_index=True)
merged = merged[["source", "date", "time", "title", "text"]]

merged.to_csv(DATA / "merged.csv", index=False)
merged.head()